# C-C network

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys
import os

date_str = "20260418"
output_folder_name = f'png_elu_{date_str}'

os.makedirs('logs_elu', exist_ok=True)
os.makedirs(f'{output_folder_name}', exist_ok=True)
os.makedirs('figs_elu', exist_ok=True)


In [ ]:
from matplotlib.patches import Patch

# plot the state space

from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from mpl_toolkits.mplot3d import Axes3D, proj3d
from matplotlib.patches import FancyArrowPatch
from matplotlib.ticker import ScalarFormatter

class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        FancyArrowPatch.__init__(self, (0,0), (0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, renderer.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        super().draw(renderer)

# b_v = 1.75
# b_h = 3.03

rho = 1
eta0 = 1
n = 1
b_v = 1.5*eta0/n
b_h = 2.25*eta0/n

# Define the vertices of the cube
def generate_vertices(rho, b_h, b_v):
    return rho*np.array([
        [0, 0, 0],
        [b_v, 0, 0],
        [b_v, b_v, 0],
        [0, b_v, 0],
        [b_h + b_v, b_h, b_h],
        [b_h + b_v, b_h + b_v, b_h],
        [0, b_h + b_v, b_h],
        [0, 0, b_h + b_v],
        [b_v, 0, b_h + b_v],
        [b_h + b_v, b_h, b_h + b_v],
        [b_h + b_v, b_h + b_v, b_h + b_v],
        [0, b_h + b_v, b_h + b_v]
    ])
    
vertices = generate_vertices(rho, b_h, b_v)

# Define the faces of the cube using the indices of vertices
faces = [
    [0, 1, 2, 3],
    [1, 2, 5, 4],
    [2, 3, 6, 5],
    [0, 1, 8, 7],
    [1, 4, 9, 8],
    [4, 5, 10, 9],
    [5, 6, 11, 10],
    [0, 3, 6, 11, 7],
    [7, 8, 9, 10, 11]
]

# Create the plot

fig = plt.figure(figsize = [6,6])
plt.rcParams.update({'font.size': 20,
                       'text.usetex': True})

# plt.rc('legend',fontsize=25)

ax = fig.add_subplot(111, projection='3d')


# Plot each face of the cube
poly3d = [[vertices[vert] for vert in face] for face in faces]
ax.add_collection3d(Poly3DCollection(poly3d, facecolors='grey', linewidths=2, edgecolors='k', alpha=0.7, zorder = 2))
ax.grid(False)
legend_handle = Patch(facecolor='grey', edgecolor='k', label='State space')

ax.set_xlabel(r'$w_1$', fontsize = 20, labelpad=0)
ax.set_ylabel(r'$w_2$', fontsize = 20, labelpad=0)
ax.set_zlabel(r'$w_3$', fontsize = 20, labelpad=0)


ax.set_xlim([-0.5*eta0/n, rho*(b_v + b_h) + 0.6*eta0/n])
ax.set_ylim([-0.5*eta0/n, rho*(b_v + b_h) + 0.6*eta0/n])
ax.set_zlim([-0.5*eta0/n, rho*(b_v + b_h) + 0.6*eta0/n])

ax.set_xticks([0, 1, 2, 3, 4])
ax.set_yticks([0, 1, 2, 3, 4])
ax.set_zticks([0, 1, 2, 3, 4])

fig.set_facecolor('white')  # Set the figure background to white
ax.set_facecolor('white')   # Set the axes background to white

# Add the arrow
a_I1 = Arrow3D([-1.2*eta0/n, -0.2*eta0/n], [(b_v + b_h)/2, (b_v + b_h)/2], [(b_v + b_h)/2, (b_v + b_h)/2], mutation_scale=20, lw=2, arrowstyle="-|>", color="r", zorder = 1)
ax.add_artist(a_I1)
a_I2 = Arrow3D([(b_v + b_h)/2, (b_v + b_h)/2], [-1.2*eta0/n, -0.2*eta0/n], [(b_v + b_h)/2, (b_v + b_h)/2], mutation_scale=20, lw=2, arrowstyle="-|>", color="b", zorder = 3)
ax.add_artist(a_I2)
a_I3 = Arrow3D([(b_v + b_h)/2, (b_v + b_h)/2], [(b_v + b_h)/2, (b_v + b_h)/2], [-1.2*eta0/n, -0.2*eta0/n], mutation_scale=20, lw=2, arrowstyle="-|>", color="g", zorder = 1)
ax.add_artist(a_I3)
a_U1 = Arrow3D([b_v + b_h + 1.2*eta0/n, b_v + b_h + 0.2*eta0/n], [(b_v + b_h)/2, (b_v + b_h)/2], [(b_v + b_h)/2, (b_v + b_h)/2], mutation_scale=20, lw=2, arrowstyle="-|>", color="magenta", zorder = 3)
ax.add_artist(a_U1)
a_U2 = Arrow3D([(b_v + b_h)/2, (b_v + b_h)/2], [b_v + b_h + 1.2*eta0/n, b_v + b_h + 0.2*eta0/n], [(b_v + b_h)/2, (b_v + b_h)/2], mutation_scale=20, lw=2, arrowstyle="-|>", color="purple", zorder = 1)
ax.add_artist(a_U2)
a_U3 = Arrow3D([(b_v + b_h)/2, (b_v + b_h)/2], [(b_v + b_h)/2, (b_v + b_h)/2], [b_v + b_h + 1.2*eta0/n, b_v + b_h + 0.2*eta0/n], mutation_scale=20, lw=2, arrowstyle="-|>", color="cyan", zorder = 3)
ax.add_artist(a_U3)
a_U4 = Arrow3D([b_v + b_h + 0.6*eta0/n, b_v + b_h + 0.1*eta0/n], [b_v + b_h + 0.6*eta0/n, b_v + b_h + 0.1*eta0/n], [b_v + b_h + 0.6*eta0/n, b_v + b_h + 0.1*eta0/n], mutation_scale=20, lw=2, arrowstyle="-|>", color="darkorange", zorder = 3)
ax.add_artist(a_U4)


# Get existing handles and labels
handles, labels = ax.get_legend_handles_labels()

# Add new handles and labels
handles.append(legend_handle)
labels.append(r'State space')

ax.legend(handles, labels, loc = 'upper right', fontsize = 18)

# ax.set_box_aspect(None, zoom=1.2)   # zoom > 1 enlarges the 3D content to fill the axes
# plt.subplots_adjust(top=0.95, bottom=0.1)


ax.tick_params(axis='x', direction='in', pad = -4)
ax.tick_params(axis='y', direction='in', pad = -2)
ax.tick_params(axis='z', direction='in', pad = 0)

plt.savefig(f"{output_folder_name}/CC3_state_space_{date_str}.png", bbox_inches='tight', pad_inches=0.05)


# proposed drift control

In [ ]:
import matplotlib.colors as mcolors

dim_K = 3
H = np.diag([80/19] * (dim_K + 1))
M_tilde_matrix = 20.0/19 * np.hstack((np.eye(dim_K, dtype=np.float64), np.ones((dim_K, 1), dtype=np.float64)))
gradient_to_drift_matrix = M_tilde_matrix @ np.linalg.inv(H) @ M_tilde_matrix.T

print(f"gradient_to_drift_matrix: {gradient_to_drift_matrix}")

alpha_S = 0.1
edge_alpha = 1
alpha_scatter = 1

font1 = {'family': 'Arial',
        'weight': 'normal',
        'size': 28}
fontsize_title = 33


def jet_to_qualitative(n_colors):
    """Convert the 'jet' colormap into a qualitative colormap with n_colors."""
    jet_cmap = plt.cm.jet_r  # Get the jet colormap
    colors = jet_cmap(np.linspace(0, 1, n_colors))  # Sample n_colors from the colormap
    return mcolors.ListedColormap(colors, name=f'jet_qualitative_{n_colors}')

# Example: Create a qualitative colormap with 8 discrete colors
n_colors = 11
qualitative_jet = jet_to_qualitative(n_colors)
# truncated_cmap = truncate_colormap(qualitative_jet, 0, 1)


azim_value = 300
model_path = "../../../1_NN_training/4_ECC_saved_NNs"
for rho_str in ['1.0']:
    for theta in ['-0.8']:
        file_name = f"RBMSteadyState_extendedCrissCross_none_100_100_100_D3_{theta}_0.0_0.01_200_0.1_64_6000_dim=3_TD1_elu_50_addcost=0.05_0.2_0.3_lr140000_eta={rho_str}_pricing=Y_delay=2.25_1.5"
        model = tf.saved_model.load(model_path + "/" + file_name + "_grad")
        cur_model_grad = keras.models.load_model(model_path + "/" + file_name + "_grad")

        rho = float(rho_str)
        
        vertices = generate_vertices(rho, b_h, b_v)
    
        grid_num = 101 
    
        x1 = np.linspace(0, rho*(b_v + b_h), grid_num, endpoint = True)
        x2 = np.linspace(0, rho*(b_v + b_h), grid_num, endpoint = True)
        x3 = np.linspace(0, rho*(b_v + b_h), grid_num, endpoint = True)
    
        X1, X2, X3 = np.meshgrid(x1, x2, x3)
    
        X1_f = X1.flatten()
        X2_f = X2.flatten()
        X3_f = X3.flatten()
    
        eps = 1e-4
        pos_s = (X2_f >= X1_f - rho*b_v - eps) & (X3_f >= X1_f - rho*b_v - eps) & (X3_f >= X2_f - rho*b_v - eps)
    
    
        X0 = np.stack((X1.reshape([-1]), X2.reshape([-1]), X3.reshape([-1])), axis = 1)
        V_gradient = model(X0, training = False, need_grad = False).numpy()
    
        V_gradient_1_mat = V_gradient[:,0]#.reshape(X1.shape)
        V_gradient_2_mat = V_gradient[:,1]#.reshape(X1.shape)
        V_gradient_3_mat = V_gradient[:,2]#.reshape(X1.shape)

        control_mat = -V_gradient @ gradient_to_drift_matrix.T * 0.5
        
        control1, control2, control3 = control_mat.T
        
        fig = plt.figure(figsize = [21,7])
        plt.rcParams.update({'font.size': 26,
                           'text.usetex': True})
    
        ax = fig.add_subplot(131, projection='3d')
    
        sc = ax.scatter(X1_f[pos_s], X2_f[pos_s], X3_f[pos_s], c=control1[pos_s], cmap=qualitative_jet, alpha = alpha_scatter, s = 20, edgecolors = 'none', marker = '.', vmin = -1.0/3, vmax = 1.0/30, zorder = 1)
    
        ax.set_xlim([0, rho * (b_v + b_h) + 0])
        ax.set_ylim([0, rho * (b_v + b_h) + 0])
        ax.set_zlim([0, rho * (b_v + b_h) + 0])
        ax.set_xlabel(r'$w_1$')
        ax.set_ylabel(r'$w_2$')
        ax.set_zlabel(r'$w_3$')
        # plt.colorbar(ticks=[0, 0.1, 0.2, 0.3])
    
        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white
    
        plt.title(r'Drift control along $w_1$', fontsize = fontsize_title)
    
        cbar = fig.colorbar(sc, pad = 0.07, ticks = [-0.3, -0.2, -0.1, 0], shrink = 0.7)
        cbar.ax.invert_yaxis()
    
        # Plot each face of the cube
        poly3d = [[vertices[vert] for vert in face] for face in faces]
        poly3d_2 = Poly3DCollection(poly3d, facecolors=(0.9,0.9,0.9), linewidths=2, edgecolors='k', alpha=alpha_S, zorder = 2)
        # Set the alpha value for the edges
        poly3d_2.set_edgecolor((0, 0, 0, edge_alpha)) 
        ax.add_collection3d(poly3d_2)
        ax.grid(False)
        legend_handle = Patch(facecolor='grey', edgecolor='k', label='State space')
        
        # Get existing handles and labels
        handles, labels = ax.get_legend_handles_labels()
        
        # Add new handles and labels
        handles.append(legend_handle)
        labels.append(r'State space')
    
        ax.view_init(elev=30, azim=azim_value)  # Change elevation and azimuth
        plt.tight_layout()
        
        # ax.legend(handles, labels, loc = 'upper left')
    
        ax = fig.add_subplot(132, projection='3d')
    
        sc = ax.scatter(X1_f[pos_s], X2_f[pos_s], X3_f[pos_s], c=control2[pos_s], cmap=qualitative_jet, alpha = alpha_scatter, s = 20, edgecolors = 'none', marker = '.', vmin = -1.0/3, vmax = 1.0/30, zorder = 1)
    
        ax.set_xlim([0, rho * (b_v + b_h) + 0])
        ax.set_ylim([0, rho * (b_v + b_h) + 0])
        ax.set_zlim([0, rho * (b_v + b_h) + 0])
        ax.set_xlabel(r'$w_1$')
        ax.set_ylabel(r'$w_2$')
        ax.set_zlabel(r'$w_3$')
    
        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white
    
        plt.title(r'Drift control along $w_2$', fontsize = fontsize_title)
        
    
        cbar = fig.colorbar(sc, pad = 0.07, ticks = [-0.3, -0.2, -0.1, 0], shrink = 0.7)
        cbar.ax.invert_yaxis()
    
        # Plot each face of the cube
        poly3d = [[vertices[vert] for vert in face] for face in faces]
        poly3d_2 = Poly3DCollection(poly3d, facecolors=(0.9,0.9,0.9), linewidths=2, edgecolors='k', alpha=alpha_S, zorder = 2)
        # Set the alpha value for the edges
        poly3d_2.set_edgecolor((0, 0, 0, edge_alpha)) 
        ax.add_collection3d(poly3d_2)
        ax.grid(False)
        legend_handle = Patch(facecolor='grey', edgecolor='k', label='State space')
        
        # Get existing handles and labels
        handles, labels = ax.get_legend_handles_labels()
        
        # Add new handles and labels
        handles.append(legend_handle)
        labels.append(r'State space')
    
        ax.view_init(elev=30, azim=azim_value)  # Change elevation and azimuth
        plt.tight_layout()
        
        # ax.legend(handles, labels, loc = 'upper left')
    
        ax = fig.add_subplot(133, projection='3d')
    
        sc = ax.scatter(X1_f[pos_s], X2_f[pos_s], X3_f[pos_s], c=control3[pos_s], cmap=qualitative_jet, alpha = alpha_scatter, s = 20, edgecolors = 'none', marker = '.', vmin = -1.0/3, vmax = 1.0/30, zorder = 1)
    
        ax.set_xlim([0, rho * (b_v + b_h) + 0])
        ax.set_ylim([0, rho * (b_v + b_h) + 0])
        ax.set_zlim([0, rho * (b_v + b_h) + 0])
        ax.set_xlabel(r'$w_1$')
        ax.set_ylabel(r'$w_2$')
        ax.set_zlabel(r'$w_3$')
    
        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white
    
        plt.title(r'Drift control along $w_3$', fontsize = fontsize_title)
        
        cbar = fig.colorbar(sc, pad = 0.07, ticks = [-0.3, -0.2, -0.1, 0], shrink = 0.7)
        cbar.ax.invert_yaxis()
    
        # Plot each face of the cube
        poly3d = [[vertices[vert] for vert in face] for face in faces]
        poly3d_2 = Poly3DCollection(poly3d, facecolors=(0.9,0.9,0.9), linewidths=2, edgecolors='k', alpha=alpha_S, zorder = 2)
        # Set the alpha value for the edges
        poly3d_2.set_edgecolor((0, 0, 0, edge_alpha)) 
        ax.add_collection3d(poly3d_2)
        ax.grid(False)
        legend_handle = Patch(facecolor='grey', edgecolor='k', label='State space')
        
        # Get existing handles and labels
        handles, labels = ax.get_legend_handles_labels()
        
        # Add new handles and labels
        handles.append(legend_handle)
        labels.append(r'State space')
    
        ax.view_init(elev=30, azim=azim_value)  # Change elevation and azimuth
        plt.tight_layout()
        
        # ax.legend(handles, labels, loc = 'upper left')
    
        plt.savefig(f"{output_folder_name}/CC3_proposed_drift_{date_str}.png", bbox_inches='tight', pad_inches=0.05)



# Proposed singular control

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tensorflow import keras
from scipy.stats import multivariate_normal as normal
import munch
import pylab
import sys
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

font1 = {'family': 'Arial',
        'weight': 'normal',
        'size': 38}

xlim_value = [-0.7,7]
ylim_value = [-0.7,6.1]

m = 20.0/19

fontsize_title = 40

for rho_str in ['1.0']:
    for theta in ['-0.8']:
        file_name = f"RBMSteadyState_extendedCrissCross_none_100_100_100_D3_{theta}_0.0_0.01_200_0.1_64_6000_dim=3_TD1_elu_50_addcost=0.05_0.2_0.3_lr140000_eta={rho_str}_pricing=Y_delay=2.25_1.5"
        model = tf.saved_model.load("../../../1_NN_training/4_ECC_saved_NNs/" + file_name + "_grad")
        cur_model_grad = keras.models.load_model("../../../1_NN_training/4_ECC_saved_NNs/" + file_name + "_grad")

        rho = float(rho_str)
        
        vertices = generate_vertices(rho, b_h, b_v)

        fig = plt.figure(figsize = [21, 15])
        plt.rcParams.update({'font.size': 38, 'text.usetex': True})
        
        ax = plt.subplot(2,3,1)

        # ############################################################
        # I2, U1
        # ############################################################
        
        steps = 400
        plot_lower = 0
        plot_upper = rho*(b_h + b_v)
        plot_slope = rho*b_v
        x1list = np.linspace(plot_lower, plot_upper, steps)
        x3list = np.linspace(plot_lower, plot_upper, steps)

        X1, X3 = np.meshgrid(x1list, x3list)

        X1_f = X1.flatten()
        X3_f = X3.flatten()
        X2_f = X1_f - plot_slope

        points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

        pos_s = (points[:,2] >= points[:,0] - plot_slope) & (points[:,2] >= points[:,1] - plot_slope) & (points[:,1] >= points[:,0] - plot_slope) & (points[:,0] >= 0) & (points[:,1] >= 0) & (points[:,2] >= 0) & (points[:,0] <= 6) & (points[:,1] <= 6) & (points[:,2] <= 6)
        # points = points[pos_s,:]

        cur_grad = cur_model_grad(points)


        Z0 = cur_grad[:,0].numpy()
        Z1 = cur_grad[:,1].numpy()
        Z2 = cur_grad[:,2].numpy()


        ax.grid(False)

        pos_I2U1 = (points[:,1] == points[:,0] - plot_slope)

        # I2
        pos = pos_I2U1 & pos_s & (Z1 < (-m*Z0 + 1)/m)  
        X1_f_0 = X1_f[pos]
        X3_f_0 = X3_f[pos]
        scatter_I2 = ax.scatter(X1_f_0,X3_f_0, c = 'b', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$L_2$')


        # U1
        pos = pos_I2U1 & pos_s & ((-m*Z0 + 1)/m < Z1)
        X1_f_0 = X1_f[pos]
        X3_f_0 = X3_f[pos]
        scatter_U1 = ax.scatter(X1_f_0,X3_f_0, c = 'magenta', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_1$')

        # draw state space
        steps = 50
        x = np.linspace(rho*b_v,rho*(b_h + b_v),steps)
        plt.plot(x, x-rho*b_v,'-k', linewidth=3, zorder = 3)
        plt.plot(x, np.ones(steps)*rho*(b_h + b_v), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*b_v, np.linspace(0,rho*(b_h + b_v),steps), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*(b_h + b_v), np.linspace(rho*b_h,rho*(b_h + b_v),steps), '-k', linewidth=3, zorder = 3)

        ax.set_xlabel(r'$w_1$', font1)
        ax.set_ylabel(r'$w_3$', font1)

        ax.set_xlim(xlim_value)
        ax.set_ylim(ylim_value)
        plt.xticks([0, 2, 4, 6])
        plt.yticks([0, 2, 4, 6])

        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white

        plt.title(r'Face $w_1 = w_2 + 1.5$', fontsize = fontsize_title)
        # plt.legend(loc = 'lower right', ncol = 1, fontsize = 33)
        # Custom legend
        custom_legend = [
            Line2D([0], [0], color='b', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$L_2$'),
            Line2D([0], [0], color='magenta', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_1$'),
        ]
        
        plt.legend(handles=custom_legend, loc = 'lower right', fontsize = 33)
        plt.tight_layout()

        ax = plt.subplot(2,3,2)

        # ############################################################
        # I3, U1
        # ############################################################
        steps = 400
        plot_lower = 0
        plot_upper = rho*(b_h + b_v)
        plot_slope = rho*b_v
        x1list = np.linspace(plot_lower, plot_upper, steps)
        x2list = np.linspace(plot_lower, plot_upper, steps)

        X1, X2 = np.meshgrid(x1list, x2list)

        X1_f = X1.flatten()
        X2_f = X2.flatten()
        X3_f = X1_f - plot_slope


        points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

        pos_s = (points[:,2] >= points[:,0] - plot_slope) & (points[:,2] >= points[:,1] - plot_slope) & (points[:,1] >= points[:,0] - plot_slope) & (points[:,0] >= 0) & (points[:,1] >= 0) & (points[:,2] >= 0) & (points[:,0] <= 6) & (points[:,1] <= 6) & (points[:,2] <= 6)
        # points = points[pos_s,:]

        cur_grad = cur_model_grad(points)


        Z0 = cur_grad[:,0].numpy()
        Z1 = cur_grad[:,1].numpy()
        Z2 = cur_grad[:,2].numpy()


        ax.grid(False)

        pos_I3U1 = (points[:,2] == points[:,0] - plot_slope)

        # I3
        pos = pos_I3U1 & pos_s & (Z2 < (-m*Z0 + 1)/m) 
        X1_f_0 = X1_f[pos]
        X2_f_0 = X2_f[pos]
        scatter_I3 = ax.scatter(X1_f_0,X2_f_0, c = 'g', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$L_3$')

        # U1
        pos = pos_I3U1 & pos_s & ((-m*Z0 + 1)/m < Z2)
        X1_f_0 = X1_f[pos]
        X2_f_0 = X2_f[pos]
        scatter_U1 = ax.scatter(X1_f_0,X2_f_0, c = 'magenta', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_1$')

        # draw state space
        steps = 50
        x = np.linspace(rho*b_v,rho*(b_v + b_h),steps)
        plt.plot(x, x-rho*b_v,'-k', linewidth=3, zorder = 3)
        plt.plot(x, x, '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*b_v, np.linspace(0,rho*b_v,steps), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*(b_v + b_h), np.linspace(rho*b_h,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)

        ax.set_xlabel(r'$w_1$', font1)
        ax.set_ylabel(r'$w_2$', font1)

        ax.set_xlim(xlim_value)
        ax.set_ylim(ylim_value)
        plt.xticks([0, 2, 4, 6])
        plt.yticks([0, 2, 4, 6])

        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white

        plt.title(r'Face $w_1 = w_3 + 1.5$', fontsize = fontsize_title)
        # plt.legend(loc = 'lower right', ncol = 1, fontsize = 33)
        # Custom legend
        custom_legend = [
            Line2D([0], [0], color='g', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$L_3$'),
            Line2D([0], [0], color='magenta', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_1$'),
        ]
        
        plt.legend(handles=custom_legend, loc = 'lower right', fontsize = 33)
        plt.tight_layout()

        ax = plt.subplot(2,3,3)

        # ############################################################
        # I3, U2
        # ############################################################
        steps = 400
        plot_lower = 0
        plot_upper = rho*(b_h + b_v)
        plot_slope = rho*b_v
        x1list = np.linspace(plot_lower, plot_upper, steps)
        x2list = np.linspace(plot_lower, plot_upper, steps)

        X1, X2 = np.meshgrid(x1list, x2list)

        X1_f = X1.flatten()
        X2_f = X2.flatten()
        X3_f = X2_f - plot_slope


        points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

        pos_s = (points[:,2] >= points[:,0] - plot_slope) & (points[:,2] >= points[:,1] - plot_slope) & (points[:,1] >= points[:,0] - plot_slope) & (points[:,0] >= 0) & (points[:,1] >= 0) & (points[:,2] >= 0) & (points[:,0] <= 6) & (points[:,1] <= 6) & (points[:,2] <= 6)
        # points = points[pos_s,:]

        cur_grad = cur_model_grad(points)


        Z0 = cur_grad[:,0].numpy()
        Z1 = cur_grad[:,1].numpy()
        Z2 = cur_grad[:,2].numpy()


        ax.grid(False)

        pos_I3U2 = (points[:,2] == points[:,1] - plot_slope)

        # I3
        pos = pos_I3U2 & pos_s & (Z2 < (-m*Z1 + 1)/m) 
        X1_f_0 = X1_f[pos]
        X2_f_0 = X2_f[pos]
        scatter_I3 = ax.scatter(X1_f_0,X2_f_0, c = 'g', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$L_3$')

        # U2
        pos = pos_I3U2 & pos_s & ((-m*Z1 + 1)/m < Z2) 
        X1_f_0 = X1_f[pos]
        X2_f_0 = X2_f[pos]
        scatter_U2 = ax.scatter(X1_f_0,X2_f_0, c = 'purple', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_2$')


        # draw state space
        steps = 50
        x = np.linspace(0,rho*(b_v + b_h),steps)
        plt.plot(x, np.maximum(rho*b_v,x),'-k', linewidth=3, zorder = 3)
        plt.plot(x, np.ones(steps)*rho*(b_v + b_h), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*0, np.linspace(rho*b_v,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)

        ax.set_xlabel(r'$w_1$', font1)
        ax.set_ylabel(r'$w_2$', font1)

        ax.set_xlim(xlim_value)
        ax.set_ylim(ylim_value)
        plt.xticks([0, 2, 4, 6])
        plt.yticks([0, 2, 4, 6])

        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white

        plt.title(r'Face $w_2 = w_3 + 1.5$', fontsize = fontsize_title)
        # plt.legend(loc = 'lower right', ncol = 1, fontsize = 33)
        # Custom legend
        custom_legend = [
            Line2D([0], [0], color='g', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$L_3$'),
            Line2D([0], [0], color='purple', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_2$'),
        ]
        
        plt.legend(handles=custom_legend, loc = 'lower right', fontsize = 33)
        plt.tight_layout()


        ax = plt.subplot(2,3,4)

        # ############################################################
        # U1U4
        # ############################################################
        steps = 400
        # upper_bound = 8
        plot_lower = 0
        plot_upper = rho*(b_h + b_v)
        plot_slope = rho*b_v
        x1list = np.linspace(plot_upper, plot_upper, 1)
        x2list = np.linspace(plot_lower, plot_upper, steps)
        x3list = np.linspace(plot_lower, plot_upper, steps)

        X1, X2, X3 = np.meshgrid(x1list, x2list, x3list)

        X1_f = X1.flatten()
        X2_f = X2.flatten()
        X3_f = X3.flatten()


        points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

        pos_s = (points[:,2] > points[:,0] - plot_slope) & (points[:,2] > points[:,1] - plot_slope) & (points[:,1] > points[:,0] - plot_slope) 
        # points = points[pos_s,:]

        cur_grad = cur_model_grad(points)


        Z0 = cur_grad[:,0].numpy()
        Z1 = cur_grad[:,1].numpy()
        Z2 = cur_grad[:,2].numpy()


        ax.grid(False)

        pos_U1U4 = (points[:,0] == plot_upper)

        # U1
        pos = pos_U1U4 & pos_s & ((-m*Z0 + 1)/m < (-m*Z0-m*Z1-m*Z2 + 1.5)/m)
        X2_f_0 = X2_f[pos]
        X3_f_0 = X3_f[pos]
        scatter_U1 = ax.scatter(X2_f_0,X3_f_0, c = 'magenta', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_1$')


        # U4
        pos = pos_U1U4 & pos_s & ((-m*Z0-m*Z1-m*Z2 + 1.5)/m < (-m*Z0 + 1)/m)
        X2_f_0 = X2_f[pos]
        X3_f_0 = X3_f[pos]
        scatter_U4 = ax.scatter(X2_f_0,X3_f_0, c = 'darkorange', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_4$')

        # draw state space
        steps = 50
        x = np.linspace(0,rho*(b_v + b_h),steps)
        plt.plot(np.linspace(rho*b_h,rho*(b_v + b_h),steps), np.ones(steps)*rho*b_h,'-k', linewidth=3, zorder = 3)
        plt.plot(np.linspace(rho*b_h,rho*(b_v + b_h),steps), np.ones(steps)*rho*(b_v + b_h), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*b_h, np.linspace(rho*b_h,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*(b_v + b_h), np.linspace(rho*b_h,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)

        ax.set_xlabel(r'$w_2$', font1)
        ax.set_ylabel(r'$w_3$', font1)

        ax.set_xlim(xlim_value)
        ax.set_ylim(ylim_value)
        plt.xticks([0, 2, 4, 6])
        plt.yticks([0, 2, 4, 6])

        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white

        plt.title(r'Face $w_1 = 3.75$', fontsize = fontsize_title)
        # plt.legend(loc = 'lower right', ncol = 1, fontsize = 33)
        # Custom legend
        custom_legend = [
            Line2D([0], [0], color='magenta', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_1$'),
            Line2D([0], [0], color='darkorange', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_4$'),
        ]
        
        plt.legend(handles=custom_legend, loc = 'lower right', fontsize = 33)
        plt.tight_layout()


        ax = plt.subplot(2,3,5)

        # ############################################################
        # U2U4
        # ############################################################
        steps = 400
        plot_lower = 0
        plot_upper = rho*(b_h + b_v)
        plot_slope = rho*b_v
        x1list = np.linspace(plot_lower, plot_upper, steps)
        x2list = np.linspace(plot_upper, plot_upper, 1)
        x3list = np.linspace(plot_lower, plot_upper, steps)

        X1, X2, X3 = np.meshgrid(x1list, x2list, x3list)

        X1_f = X1.flatten()
        X2_f = X2.flatten()
        X3_f = X3.flatten()


        points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

        pos_s = (points[:,2] > points[:,0] - plot_slope) & (points[:,2] > points[:,1] - plot_slope) & (points[:,1] > points[:,0] - plot_slope) 
        # points = points[pos_s,:]

        cur_grad = cur_model_grad(points)


        Z0 = cur_grad[:,0].numpy()
        Z1 = cur_grad[:,1].numpy()
        Z2 = cur_grad[:,2].numpy()


        ax.grid(False)

        pos_U2U4 = (points[:,1] == plot_upper)

        # U2
        pos = pos_U2U4 & pos_s & ((-m*Z1 + 1)/m < (-m*Z0-m*Z1-m*Z2 + 1.5)/m)  
        X1_f_0 = X1_f[pos]
        X3_f_0 = X3_f[pos]
        scatter_U2 = ax.scatter(X1_f_0,X3_f_0, c = 'purple', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_2$')

        # U4
        pos = pos_U2U4 & pos_s & ((-m*Z0-m*Z1-m*Z2 + 1.5)/m < (-m*Z1 + 1)/m)  
        X1_f_0 = X1_f[pos]
        X3_f_0 = X3_f[pos]
        scatter_U4 = ax.scatter(X1_f_0,X3_f_0, c = 'darkorange', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_4$')

        # draw state space
        steps = 50
        x = np.linspace(0,rho*(b_v + b_h),steps)
        plt.plot(np.linspace(0,rho*(b_v + b_h),steps), np.ones(steps)*rho*b_h,'-k', linewidth=3, zorder = 3)
        plt.plot(np.linspace(0,rho*(b_v + b_h),steps), np.ones(steps)*rho*(b_v + b_h), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*0, np.linspace(rho*b_h,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*(b_v + b_h), np.linspace(rho*b_h,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)

        ax.set_xlabel(r'$w_1$', font1)
        ax.set_ylabel(r'$w_3$', font1)

        ax.set_xlim(xlim_value)
        ax.set_ylim(ylim_value)
        plt.xticks([0, 2, 4, 6])
        plt.yticks([0, 2, 4, 6])

        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white

        plt.title(r'Face $w_2 = 3.75$', fontsize = fontsize_title)
        # plt.legend(loc = 'lower right', ncol = 1, fontsize = 33)
        # Custom legend
        custom_legend = [
            Line2D([0], [0], color='purple', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_2$'),
            Line2D([0], [0], color='darkorange', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_4$'),
        ]
        
        plt.legend(handles=custom_legend, loc = 'lower right', fontsize = 33)
        plt.tight_layout()


        ax = plt.subplot(2,3,6)

        # ############################################################
        # U3U4
        # ############################################################
        steps = 400
        plot_lower = 0
        plot_upper = rho*(b_h + b_v)
        plot_slope = rho*b_v
        x1list = np.linspace(plot_lower, plot_upper, steps)
        x2list = np.linspace(plot_lower, plot_upper, steps)
        x3list = np.linspace(plot_upper, plot_upper, 1)

        X1, X2, X3 = np.meshgrid(x1list, x2list, x3list)

        X1_f = X1.flatten()
        X2_f = X2.flatten()
        X3_f = X3.flatten()


        points = np.concatenate((X1_f.reshape(1,-1), X2_f.reshape(1,-1), X3_f.reshape(1,-1)), axis=0).transpose()

        pos_s = (points[:,2] > points[:,0] - plot_slope) & (points[:,2] > points[:,1] - plot_slope) & (points[:,1] > points[:,0] - plot_slope) 
        # points = points[pos_s,:]

        cur_grad = cur_model_grad(points)


        Z0 = cur_grad[:,0].numpy()
        Z1 = cur_grad[:,1].numpy()
        Z2 = cur_grad[:,2].numpy()

        ax.grid(False)

        pos_U3U4 = (points[:,2] == plot_upper)
        # U3
        pos = pos_U3U4 & pos_s & ((-m*Z2 + 1)/m < (-m*Z0-m*Z1-m*Z2 + 1.5)/m)
        X1_f_0 = X1_f[pos]
        X2_f_0 = X2_f[pos]
        scatter_U3 = ax.scatter(X1_f_0,X2_f_0, c = 'cyan', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_3$')


        # U4
        pos = pos_U3U4 & pos_s & ((-m*Z0-m*Z1-m*Z2 + 1.5)/m < (-m*Z2 + 1)/m)
        X1_f_0 = X1_f[pos]
        X2_f_0 = X2_f[pos]
        scatter_U4 = ax.scatter(X1_f_0,X2_f_0, c = 'darkorange', marker = '.', alpha = 1, s = 5, zorder = 2, label = r'$U_4$')

        # draw state space
        steps = 50
        x = np.linspace(0,rho*(b_v + b_h),steps)
        plt.plot(x, np.maximum(0,x-rho*b_v),'-k', linewidth=3, zorder = 3)
        plt.plot(x, np.ones(steps)*rho*(b_v + b_h), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*0, np.linspace(0,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)
        plt.plot(np.ones(steps)*rho*(b_v + b_h), np.linspace(rho*b_h,rho*(b_v + b_h),steps), '-k', linewidth=3, zorder = 3)

        ax.set_xlabel(r'$w_1$', font1)
        ax.set_ylabel(r'$w_2$', font1)

        ax.set_xlim(xlim_value)
        ax.set_ylim(ylim_value)
        plt.xticks([0, 2, 4, 6])
        plt.yticks([0, 2, 4, 6])

        fig.set_facecolor('white')  # Set the figure background to white
        ax.set_facecolor('white')   # Set the axes background to white

        plt.title(r'Face $w_3 = 3.75$', fontsize = fontsize_title)
        # plt.legend(loc = 'lower right', ncol = 1, fontsize = 33)
        # Custom legend
        custom_legend = [
            Line2D([0], [0], color='cyan', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_3$'),
            Line2D([0], [0], color='darkorange', marker='.', markersize=20, linestyle='', linewidth = 3, label=r'$U_4$'),
        ]
        
        plt.legend(handles=custom_legend, loc = 'lower right', fontsize = 33)
        plt.tight_layout()

        plt.savefig(f"{output_folder_name}/CC3_proposed_singular_{date_str}.png", bbox_inches='tight', pad_inches=0.05)

